# Project 74 — Style Dataset Creator
## Extract Writing Style → Generate Style-Matched Training Data

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Style Samples

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.7)

style_samples = {
    "technical": [
        "The microservices architecture employs gRPC for inter-service communication with "
        "Protocol Buffers serialization, achieving sub-millisecond latency.",
        "Database sharding distributes data across partitions using consistent hashing, "
        "ensuring horizontal scalability with minimal rebalancing overhead.",
    ],
    "casual": [
        "So basically we split everything into tiny services that talk to each other. "
        "It's super fast and way easier to debug than the old monolith.",
        "We spread the data across multiple databases so nothing gets too big. "
        "Pretty slick setup honestly!",
    ],
    "executive": [
        "Our platform modernization reduced operational costs by 35% while improving "
        "system reliability to 99.95% uptime, positioning us for 10x scale.",
        "Strategic investment in AI capabilities has yielded 40% faster time-to-market "
        "and a 25% improvement in customer satisfaction scores.",
    ],
}
print(f"Style categories: {list(style_samples.keys())}")

## Step 2 — Analyze Style DNA

In [ ]:
class StyleDNA(BaseModel):
    tone: str
    formality: int = Field(ge=1, le=5, description="1=casual, 5=formal")
    avg_sentence_length: str = Field(description="short, medium, long")
    vocabulary_level: str = Field(description="simple, intermediate, advanced, technical")
    uses_jargon: bool
    uses_metrics: bool
    distinctive_patterns: list[str]

analyzer = llm.with_structured_output(StyleDNA)

profiles = {}
for style, samples in style_samples.items():
    combined = "\n".join(samples)
    profile = analyzer.invoke(f"Analyze the writing style of these samples:\n{combined}")
    profiles[style] = profile
    print(f"\n{style.upper()} STYLE DNA:")
    print(f"  Tone: {profile.tone}")
    print(f"  Formality: {profile.formality}/5")
    print(f"  Vocabulary: {profile.vocabulary_level}")
    print(f"  Jargon: {profile.uses_jargon} | Metrics: {profile.uses_metrics}")
    print(f"  Patterns: {profile.distinctive_patterns}")

## Step 3 — Generate Style-Matched Training Data

In [ ]:
topics = [
    "cloud computing benefits",
    "machine learning model deployment",
    "team productivity improvement",
    "data security best practices",
]

gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write about the topic in the specified style. "
     "Match the style DNA precisely: tone, formality, vocabulary."),
    ("human", "Style: {style}\nStyle DNA: {dna}\nTopic: {topic}\n\nGenerate 2 paragraphs:")
])
gen_chain = gen_prompt | llm | StrOutputParser()

dataset = []
for style, profile in profiles.items():
    for topic in topics:
        text = gen_chain.invoke({
            "style": style,
            "dna": json.dumps(profile.model_dump()),
            "topic": topic,
        })
        dataset.append({
            "style": style, "topic": topic,
            "text": text, "formality": profile.formality,
        })
        print(f"  {style}/{topic}: {len(text)} chars")

df = pd.DataFrame(dataset)
print(f"\nDataset: {len(df)} style-matched samples")
print(df.groupby("style").size().to_string())

## Step 4 — Style Consistency Verification

In [ ]:
# Cross-check: can the LLM correctly identify the style?
verify_prompt = ChatPromptTemplate.from_template(
    "What writing style is this: technical, casual, or executive?\n\n{text}\n\nStyle:"
)
verify_chain = verify_prompt | llm | StrOutputParser()

correct = 0
for _, row in df.iterrows():
    predicted = verify_chain.invoke({"text": row["text"][:300]}).strip().lower()
    match = row["style"] in predicted
    correct += match

consistency = correct / len(df) if len(df) > 0 else 0
print(f"Style consistency: {consistency:.0%} ({correct}/{len(df)})")

# Save dataset
df.to_json("sample_data/style_training_data.json", orient="records", indent=2)
print("✓ Saved to sample_data/style_training_data.json")

## What You Learned
- **Style DNA extraction** with quantified attributes
- **Style-conditioned text generation** for training data
- **Consistency verification** via round-trip classification
- **Training dataset export** for style fine-tuning